# LifeMate baseline — Qwen3-4B zero-shot trên transcript gốc

1. Trên máy local: `python pack.py` → tạo `dist/lifemate_code.zip` và `dist/zaic_data_noaudio.zip`.
2. Upload 2 file zip vào Google Drive `MyDrive/zaloai/` (giữ private — dữ liệu BTC không được phát tán).
3. Runtime → GPU (T4 là đủ; L4/A100 nhanh hơn). Chạy lần lượt các cell.

Trên Kaggle: upload 2 zip thành một private Dataset rồi sửa `SRC` ở cell 2 thành `/kaggle/input/<tên-dataset>`.

In [ ]:
!nvidia-smi

In [ ]:
import os, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC = '/content/drive/MyDrive/zaloai'
    WORK = '/content/work'
except ImportError:  # Kaggle
    SRC = '/kaggle/input/zaloai'
    WORK = '/kaggle/working/work'
os.makedirs(WORK + '/data', exist_ok=True)
zipfile.ZipFile(f'{SRC}/lifemate_code.zip').extractall(WORK)
zipfile.ZipFile(f'{SRC}/zaic_data_noaudio.zip').extractall(WORK + '/data')
os.chdir(WORK)
os.environ['LIFEMATE_DATA'] = WORK + '/data'
print(os.listdir(WORK), os.listdir(WORK + '/data'))

In [ ]:
!pip install -q vllm

In [ ]:
# sanity: oracle phải ra SCORE 1.0000
!python -m lifemate.run --split train --ids val_seen --backend oracle

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
!python -m lifemate.run --split train --ids val_seen --backend vllm --model {MODEL} --out runs/val_seen_qwen3_4b

In [ ]:
!python -m lifemate.run --split train --ids val_unseen --backend vllm --model {MODEL} --out runs/val_unseen_qwen3_4b

In [ ]:
# phân tích lỗi nhanh: mẫu điểm 0 theo nhóm
import json, collections
for run in ['runs/val_seen_qwen3_4b', 'runs/val_unseen_qwen3_4b']:
    rows = [json.loads(l) for l in open(f'{run}/scores.jsonl')]
    print(run, collections.Counter((r['category'], 'hash_ok' if r['hash_ok'] else 'hash_fail') for r in rows if r['score'] == 0))

In [ ]:
# lưu kết quả về Drive / output
import shutil
shutil.make_archive(f'{SRC if SRC.startswith("/content") else "/kaggle/working"}/runs_baseline', 'zip', 'runs')